In [3]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
import evaluate
import os
import torch

In [5]:
mls = load_dataset("facebook/multilingual_librispeech", "french", split="dev", streaming="true")

In [6]:
print(mls.features)

{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'original_path': Value('string'), 'begin_time': Value('float64'), 'end_time': Value('float64'), 'transcript': Value('string'), 'audio_duration': Value('float64'), 'speaker_id': Value('string'), 'chapter_id': Value('string'), 'file': Value('string'), 'id': Value('string')}


In [7]:
mls = mls.cast_column("audio", Audio(sampling_rate=16_000))

In [8]:
input_speech = next(iter(mls))

In [9]:
signal_audio = input_speech['audio']['array']

In [10]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

In [11]:
input_features = processor( signal_audio, sampling_rate=input_speech['audio']["sampling_rate"], return_tensors="pt").input_features

In [16]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
forced_decoder_ids = processor.get_decoder_prompt_ids(language="fr", task="transcribe")

In [17]:
predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

In [14]:
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
label = input_speech['transcript']

In [15]:
print("whisper: ",transcription)
print("la verite: ",label)

whisper:   But he sent him to find the fisherman himself, and when he arrived, the fisherman said to him, Give me four or three fish that make it seem to those you have already brought, because he is overcome by a certain evil that has prevented us from serving them at the same time.
la verite:  cependant il envoya chercher le pêcheur à l'heure même et quand il fut arrivé pêcheur lui dit-il apporte-moi quatre autres poissons qui soient semblables à ceux que tu as déjà apportés car il est survenu un certain malheur qui a empêché qu'on ne les ait servis au sultan
